In [1]:
%load_ext autoreload
%autoreload 2
%config IPCompleter.greedy = True

# Tutorial 2: Dissipative Participation Ratios and Dielectric Loss

Author: Zlatko Minev

> **Series navigation:** [← Tutorial 1: Startup Example](Tutorial%201.%20%20Startup%20example.ipynb) &nbsp;|&nbsp; [Tutorial 3: Circuit QED Parameters →](Tutorial%203.%20%20toolbox_circuits.ipynb)

## Introduction: Dielectric Loss in Superconducting Qubits

One of the central challenges in building high-coherence superconducting qubits is minimising energy loss to the electromagnetic environment. Among the dominant loss channels is **dielectric loss**: energy absorbed by lossy dielectric materials (substrate, surface oxides, resist residues) that are threaded by the electric field of the qubit mode.

### The dissipative EPR

The **energy-participation ratio (EPR) method** generalises naturally to dissipative elements. For a lossy dielectric volume $V$ with loss tangent $\tan\delta$, the dielectric loss rate of mode $m$ is:

$$\kappa_m^{(\mathrm{diel})} = \sum_k p_k^{(\mathrm{diel})} \cdot \tan\delta_k \cdot \omega_m$$

where the **dielectric participation ratio** is:

$$p_k^{(\mathrm{diel})} = \frac{\mathcal{E}_{\mathrm{elec},k}}{\mathcal{E}_{\mathrm{elec},\mathrm{total}}}$$

This is the fraction of the mode's electric energy stored in volume $k$. A small $p_k$ means the dielectric barely interacts with the qubit mode; reducing $p_k$ (by geometry optimisation) directly improves coherence time $T_1 = 1/\kappa$.

### Energy definitions (from Appendix E of [arXiv:2010.00620](https://arxiv.org/abs/2010.00620))

The time-averaged electric and magnetic energies stored in volume $V$ are:

$$\mathcal{E}_{\mathrm{elec}} = \frac{1}{4}\,\mathrm{Re}\int_V \mathrm{d}v\;\vec{E}_{\max}^* \cdot \overleftrightarrow{\epsilon} \cdot \vec{E}_{\max}$$

$$\mathcal{E}_{\mathrm{mag}} = \frac{1}{4}\,\mathrm{Re}\int_V \mathrm{d}v\;\vec{H}_{\max}^* \cdot \overleftrightarrow{\mu} \cdot \vec{H}_{\max}$$

where $\vec{E}_{\max}$ and $\vec{H}_{\max}$ are the complex peak-amplitude field distributions from the HFSS eigenmode solution. pyEPR computes these integrals by invoking the HFSS fields calculator through its COM interface.

### Typical loss channels

| Channel | Location | Typical $p$ | $\tan\delta$ |
|---|---|---|---|
| Bulk substrate (Si, sapphire) | Volume | 5–40 % | $\sim 10^{-6}$ |
| Substrate surface (native oxide) | Interface sheet | 0.1–2 % | $\sim 10^{-3}$ |
| Metal–substrate interface | Sheet | 0.1–5 % | $\sim 10^{-3}$ |
| Seam (gap in ground plane) | Line | — | current-dependent |

Reducing the qubit's electric-field participation in the substrate is a major driver of 3D cavity architectures, flip-chip designs, and substrate etching strategies.

**Reference:** Z. K. Minev *et al.*, *Energy-participation quantization of Josephson circuits*, [npj Quantum Information **7**, 131 (2021)](https://doi.org/10.1038/s41534-021-00461-8) ([arXiv:2010.00620](https://arxiv.org/abs/2010.00620)), Appendix E.

## The simple way

Following Tutorial 1, let us load the `pyEPR` package and connect to the example HFSS design. We will then compute the dielectric participation of the silicon substrate at three levels of abstraction, from the highest-level convenience function down to direct use of the HFSS fields calculator.

In [2]:
import pyEPR_pyaedt as epr

#### Load the Ansys HFSS tutorial file

As in Tutorial 1, let us find the path to the example project.

In [3]:
from pathlib import Path
path_to_project = Path(epr.__file__).parent.parent / '_example_files'
print(f'Example project located in:\n {path_to_project}')

Example project located in:
 C:\Code\pyEPR_pyaedt\_example_files


Connect to HFSS and create the distributed-analysis object `eprh`:

In [4]:
pinfo = epr.ProjectInfo(project_path = path_to_project, 
                         project_name = 'pyEPR_tutorial1',
                         design_name  = '1. single_transmon')
eprh = epr.DistributedAnalysis(pinfo)

INFO 01:19PM [connect_project]: Connecting to Ansys Desktop API...


INFO 01:19PM [load_ansys_project]: 	File path to HFSS project found.


WARNING 01:19PM [load_ansys_project]: 		File is locked. 😨 If connection fails, delete the .lock file.


INFO 01:19PM [connect_desktop]: Connecting to AEDT via PyAEDT (version=None, new_desktop=False, non_graphical=False)...


PyAEDT INFO: Python version 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)].


...

INFO 01:19PM [connect_design]: 	Opened active design
	Design:    1. single_transmon [Solution type: Eigenmode]


INFO 01:19PM [get_setup]: 	Opened setup `Setup1`  (<class 'pyEPR_pyaedt.ansys.HfssEMSetup'>)


INFO 01:19PM [connect]: 	Connected to project "pyEPR_tutorial1" and design "1. single_transmon" 😀 



INFO 01:19PM [__init__]: Design "1. single_transmon" info:


INFO 01:19PM [__init__]: 	# eigenmodes    2
	# variations    6


## Calculating the substrate participation for mode 0 (the qubit mode)

The fundamental eigenmode (mode index 0) is the qubit mode — a standing wave with maximum electric field concentrated near the Josephson junction and the capacitor pads. We first select this mode and then ask: what fraction of the mode's total electric energy is stored in the silicon substrate?

This single number, $p_{\mathrm{Si}}$, directly determines how strongly the qubit loses energy to substrate two-level-system (TLS) defects:
$$T_1^{-1}(\text{substrate TLS}) \approx p_{\mathrm{Si}} \cdot \tan\delta_{\mathrm{Si}} \cdot \omega_q$$

In [5]:
eprh.set_mode(0) 

# Calculate the dielectric energy-participation ratio of the substrate
# relative to the total electric energy of all objects.
#
# Returns: p_dielectric, (E_object, E_total)
p_dielectric, (E_substr, E_total) = eprh.calc_p_electric_volume('substrate', 'AllObjects')

print(f'Energy in silicon substrate  = {100*p_dielectric:.1f}%')

Energy in silicon substrate  = 87.3%


Now compute the electric energy stored in the vacuum (cavity enclosure). We pass in the already-computed `E_total` to avoid recomputing it — field integrals in HFSS can be slow for large meshes.

In [6]:
# Pass in the precomputed E_total so we don't recompute it
p_vac, (E_vac, E_total) = eprh.calc_p_electric_volume('cavity_enclosure', E_total=E_total)

print(f'Energy in vacuum = {100*p_vac:.1f}%')
print(f'\nSince there are no other volumes, the two should sum to one: {p_dielectric + p_vac:.6f}')

Energy in vacuum = 12.7%

Since there are no other volumes, the two should sum to one: 1.000000


Let us inspect the full function signature. Note the `E_total` optional argument, useful for caching expensive field integrals:

In [7]:
? eprh.calc_p_electric_volume

## Level 2: Calculating the energies directly

The `calc_p_electric_volume` convenience function internally calls `calc_energy_electric`, which performs a single volume integral of $\vec{E}^* \cdot \overleftrightarrow{\epsilon} \cdot \vec{E}$ using the HFSS fields calculator. We can call this directly:

In [8]:
E_total  = eprh.calc_energy_electric(obj='AllObjects')
E_substr = eprh.calc_energy_electric(obj='substrate')
print(f'Energy in substrate = {100*E_substr/E_total:.1f}%')

Energy in substrate = 87.3%


In [9]:
?eprh.calc_energy_electric

## Level 3: Using the HFSS Fields Calculator directly

For complete transparency and for custom field integrals not built into pyEPR, you can use the HFSS fields calculator object `CalcObject` directly. This is also useful for computing surface integrals (seam loss), line integrals (current density), or any other derived field quantity.

### The HFSS Fields Calculator

The HFSS Fields Calculator enables post-processing of arbitrary derived quantities from the EM field solution. It operates lazily — the computation is deferred until a numerical result is requested — which makes it efficient for chaining operations. The calculator is accessed via pyEPR's `CalcObject` wrapper.

### Direct computation of the total electric energy

We compute:
$$\mathcal{E}_{\mathrm{elec}} = \mathrm{Re}\int_V \mathrm{d}v\;\vec{E}_{\max}^* \cdot \overleftrightarrow{\epsilon} \cdot \vec{E}_{\max}$$

step by step: retrieve $\vec{E}$, smooth it, form $\vec{E}^* \cdot \epsilon \cdot \vec{E}$, take the real part, and integrate over the volume. Each step pushes an operation onto the calculator stack.

In [10]:
from pyEPR_pyaedt.core import *
from pyEPR_pyaedt.ansys import CalcObject

self, volume = eprh, 'AllObjects'

calcobject = CalcObject([], self.setup)
vecE = calcobject.getQty("E").smooth()
A = vecE.times_eps()
B = vecE.conj()
A = A.dot(B)
A = A.real()
A = A.integrate_vol(name=volume)

E_total = A.evaluate(lv=self._get_lv()) 
E_total

4.30737313672182e-22

In [11]:
from pyEPR_pyaedt.core import *

self, volume = eprh, 'substrate'

calcobject = CalcObject([], self.setup)
vecE = calcobject.getQty("E").smooth()
A = vecE.times_eps()
B = vecE.conj()
A = A.dot(B)
A = A.real()
A = A.integrate_vol(name=volume)

E_subs = A.evaluate(lv=self._get_lv()) 
E_subs

3.80373110124248e-22

In [12]:
print(f'Energy in substrate: {100*E_subs/E_total:.1f}%')

Energy in substrate: 88.3%


## Summary

We demonstrated three equivalent paths to compute the dielectric participation ratio $p_{\mathrm{Si}}$, in increasing order of transparency:

| Level | Function | Best for |
|---|---|---|
| High | `eprh.calc_p_electric_volume(...)` | Quick participation of standard volumes |
| Mid | `eprh.calc_energy_electric(obj=...)` | Custom volume ratios, reusing `E_total` |
| Low | `CalcObject` fields calculator directly | Custom integrals, surface/line loss, debugging |

All three give the same answer. The choice depends on how much control you need over the calculation.

### Next steps

With the participation ratios in hand, you can compute the **dielectric quality factor**:
$$Q_m^{(\mathrm{diel})} = \frac{1}{p_{\mathrm{Si}} \cdot \tan\delta_{\mathrm{Si}}}$$

For silicon, $\tan\delta \approx 5 \times 10^{-7}$ at millikelvin temperatures and low fields. A participation of 10 % would give $Q \approx 2 \times 10^7$, or $T_1 \approx 500\,\mu s$ at 5 GHz — consistent with state-of-the-art 3D transmon experiments.

Surface and seam losses can be computed analogously using `calc_p_electric_surface` and related functions. See the pyEPR documentation for the full API.

> **Reference:** Appendix E of [Minev *et al.*, arXiv:2010.00620](https://arxiv.org/abs/2010.00620).

---
> **Up next:** [Tutorial 3 — Circuit QED Parameters: Josephson Junctions, Transmons, and Unit Conversions](Tutorial%203.%20%20toolbox_circuits.ipynb)